### Теперь займемся подготовкой нашего финального датасета, путем мерджа спаршенного датасета со остальными, полученными через IGDB API

Импортируем необходимые библиотеки:

In [1]:
import pandas as pd

Далее, прочтем все имеющиеся датасеты для дальйнешей их обработки и конечного мерджа

In [2]:
df_parsed = pd.read_excel("../../data/df_parsed.xlsx")

In [3]:
df_games_final = pd.read_csv("../../data/df_games_final.csv")
df_popular_final = pd.read_csv("../../data/df_popular_final.csv")
df_web_final = pd.read_csv("../../data/df_web_final.csv")
igdb_steam_games_final = pd.read_csv("../../data/igdb_steam_games_final.csv")

print(df_parsed.columns)
print(df_games_final.columns)
print(df_popular_final.columns)
print(df_web_final.columns)
print(igdb_steam_games_final.columns)

Index(['steam_id', 'game_name', 'game_description_snippet', 'game_price',
       'found_game_price', 'all_language_reviews_type',
       'all_language_reviews_count', 'has_russian_reviews',
       'all_russian_reviews_type', 'all_russian_reviews_count',
       'steam_app_url', 'support_ru_region'],
      dtype='str')
Index(['id', 'first_release_date', 'name', 'rating', 'rating_count',
       'total_rating', 'total_rating_count'],
      dtype='str')
Index(['id', 'game_id', 'value', 'external_popularity_source'], dtype='str')
Index(['id', 'game', 'type'], dtype='str')
Index(['uid', 'name', 'id', 'game'], dtype='str')


Для начала сделаем merge нашей основной таблицы df_parsed с igdb_steam_games_final. Поскольку в будущем, нам нужно будет делать merge с таблицами для обогащения, полученных с api запросов (df_games_finalб df_popular_final, df_web_final). Нам нужно добавить id игр с igbd в df_parsed, чтобы в дальнейшем осуществить merge с остальными. 

Проверим пустые значения, уникальные значения и дубли перед мерджем

In [4]:
print(df_parsed["steam_id"].isna().sum())
print(df_parsed["steam_id"].nunique())
print(df_parsed.duplicated(subset=["steam_id"]).sum())

print(igdb_steam_games_final["uid"].isna().sum())
print(igdb_steam_games_final["uid"].nunique())
print(igdb_steam_games_final.duplicated(subset=["uid"]).sum())

0
26853
0
0
25699
1


In [5]:
igdb_steam_games_final[igdb_steam_games_final["uid"].duplicated(keep=False)]

,uid,name,id,game
2890,1438480,Saviorless,2451784.0,25338
2891,1438480,Saviorless,2048921.0,25338


In [6]:
igdb_steam_games_final = igdb_steam_games_final.drop(columns = {"id"})

In [7]:
igdb_steam_games_final

,uid,name,game
0,789570,Nepenthe,100600
1,1424520,Astria,169971
2,751590,Dinosaurs A Prehistoric Adventure,57128
3,467850,METAGAL,19321
4,1717570,Помпом,173842
...,...,...,...
25695,302570,Dangerous,17429
25696,911130,Puzzle Monarch: Vampires,106446
25697,347820,Street Arena,35817
25698,1212040,Paws & Effect: My Dogs Are Human!,128455


Нашли дубликат, проверили, эта одна и та же игра с одинаковыми id на igdb и steam, разные только id, оставшийся с 
таблицы(при сохранении не прописали index = False). Его дропнули по причине ненадобности, оставшиеся данные нужны,
по ним будем осуществлять первый мердж (по uid (steam id)), и в дальнейшем по game(igdb id). 

Удалим дубликат:

In [8]:
igdb_steam_games_final = igdb_steam_games_final.drop_duplicates()
igdb_steam_games_final[igdb_steam_games_final["uid"].duplicated(keep=False)]

,uid,name,game


In [9]:
print(igdb_steam_games_final["uid"].isna().sum())
print(igdb_steam_games_final["uid"].nunique())
print(igdb_steam_games_final.duplicated(subset=["uid"]).sum())

0
25699
0


Отлично, приступаем в merge

In [10]:
df_parsed_1 =df_parsed.merge(igdb_steam_games_final, left_on="steam_id", right_on="uid",how="inner")
df_parsed_1

,steam_id,game_name,game_description_snippet,game_price,found_game_price,all_language_reviews_type,all_language_reviews_count,has_russian_reviews,all_russian_reviews_type,all_russian_reviews_count,steam_app_url,support_ru_region,uid,name,game
0,1313940,My Holiday 2,This is the second part of the long holiday of...,259 руб.,1.0,Очень положительные,106.0,0.0,NaN,NaN,https://store.steampowered.com/app/1313940/,1.0,1313940,My Holiday 2,150866
1,1902870,Vertical Kingdom,Возродите былое величие Империи в этой карточн...,710 руб.,1.0,В основном положительные,111.0,0.0,NaN,NaN,https://store.steampowered.com/app/1902870/,1.0,1902870,Vertical Kingdom,202602
2,1071940,Streamer's Life,"Игра в жанре RPG-Simulation, в которой вам пре...",NaN,0.0,В основном положительные,139.0,0.0,NaN,NaN,https://store.steampowered.com/app/1071940/,1.0,1071940,Streamer's Life,124447
3,1706570,Chupacabras: Night Hunt,Hunt the chupacabras alone or with your friend...,99 руб.,1.0,В основном положительные,124.0,0.0,NaN,NaN,https://store.steampowered.com/app/1706570/,1.0,1706570,Chupacabras: Night Hunt,163904
4,1085150,Golf Defied,"Физика и механики нового поколения, мультиплее...",Бесплатные,1.0,Смешанные,175.0,0.0,NaN,NaN,https://store.steampowered.com/app/1085150/,1.0,1085150,Golf Defied,118634
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25694,1547900,Active Mummy,"Active Mummy — динамичный 2D-платформер, в кот...",210 руб.,1.0,Положительные,20.0,0.0,NaN,NaN,https://store.steampowered.com/app/1547900/,1.0,1547900,Active Mummy,156747
25695,2238360,You Have No Time,Rev up your inner speedster in this high-octan...,165 руб.,1.0,Положительные,13.0,0.0,NaN,NaN,https://store.steampowered.com/app/2238360/,1.0,2238360,You Have No Time,244854
25696,1490030,Cyber Dodge,Cyber Dodge is a skill-based Runner and a Die ...,61 руб.,1.0,Обзоров пользователей: 3,3.0,0.0,NaN,NaN,https://store.steampowered.com/app/1490030/,1.0,1490030,Cyber Dodge,159989
25697,1855190,Vektor Z,Vektor Z brings fast and furious retro space b...,280 руб.,1.0,Обзоров пользователей: 2,2.0,0.0,NaN,NaN,https://store.steampowered.com/app/1855190/,1.0,1855190,Vektor Z,186672


Видим датасет получившийся, нужно убрать uid (дублирует steam_id), name (дублирует game_name), и для наглядности переименуем game в igdb_id

In [11]:
df_parsed_1 = df_parsed_1.drop(columns = {"name", "uid"})
df_parsed_1 = df_parsed_1.rename(columns = {"game": "igdb_id"})
df_parsed_1

,steam_id,game_name,game_description_snippet,game_price,found_game_price,all_language_reviews_type,all_language_reviews_count,has_russian_reviews,all_russian_reviews_type,all_russian_reviews_count,steam_app_url,support_ru_region,igdb_id
0,1313940,My Holiday 2,This is the second part of the long holiday of...,259 руб.,1.0,Очень положительные,106.0,0.0,NaN,NaN,https://store.steampowered.com/app/1313940/,1.0,150866
1,1902870,Vertical Kingdom,Возродите былое величие Империи в этой карточн...,710 руб.,1.0,В основном положительные,111.0,0.0,NaN,NaN,https://store.steampowered.com/app/1902870/,1.0,202602
2,1071940,Streamer's Life,"Игра в жанре RPG-Simulation, в которой вам пре...",NaN,0.0,В основном положительные,139.0,0.0,NaN,NaN,https://store.steampowered.com/app/1071940/,1.0,124447
3,1706570,Chupacabras: Night Hunt,Hunt the chupacabras alone or with your friend...,99 руб.,1.0,В основном положительные,124.0,0.0,NaN,NaN,https://store.steampowered.com/app/1706570/,1.0,163904
4,1085150,Golf Defied,"Физика и механики нового поколения, мультиплее...",Бесплатные,1.0,Смешанные,175.0,0.0,NaN,NaN,https://store.steampowered.com/app/1085150/,1.0,118634
...,...,...,...,...,...,...,...,...,...,...,...,...,...
25694,1547900,Active Mummy,"Active Mummy — динамичный 2D-платформер, в кот...",210 руб.,1.0,Положительные,20.0,0.0,NaN,NaN,https://store.steampowered.com/app/1547900/,1.0,156747
25695,2238360,You Have No Time,Rev up your inner speedster in this high-octan...,165 руб.,1.0,Положительные,13.0,0.0,NaN,NaN,https://store.steampowered.com/app/2238360/,1.0,244854
25696,1490030,Cyber Dodge,Cyber Dodge is a skill-based Runner and a Die ...,61 руб.,1.0,Обзоров пользователей: 3,3.0,0.0,NaN,NaN,https://store.steampowered.com/app/1490030/,1.0,159989
25697,1855190,Vektor Z,Vektor Z brings fast and furious retro space b...,280 руб.,1.0,Обзоров пользователей: 2,2.0,0.0,NaN,NaN,https://store.steampowered.com/app/1855190/,1.0,186672


Перейдем к следующему датасету (df_web_final) и присоединим к общему колонку type, которая в свою очередь показывае, на какой платформе или каком сайте у игры есть страница.

In [12]:
print(df_web_final.isna().sum())
print(df_web_final.duplicated().sum()) #видим 10 дублей, посмотрим их

id      0
game    0
type    0
dtype: int64
10


In [13]:
df_web_final[df_web_final.duplicated(keep=False)] 


,id,game,type
1999,895790,135119,YouTube
2064,170916,138871,GOG
2065,159645,138871,Community Wiki
2066,155929,138871,Steam
2067,155930,138871,Official Website
2069,800116,33027,Nintendo
2396,169267,138871,Epic
4023,170916,138871,GOG
4024,159645,138871,Community Wiki
4025,155929,138871,Steam


In [14]:
df_web_final = df_web_final.drop_duplicates()
print(df_web_final[df_web_final.duplicated(keep=False)])
df_web_final.info()

Empty DataFrame
Columns: [id, game, type]
Index: []
<class 'pandas.DataFrame'>
Index: 25990 entries, 0 to 25999
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   id      25990 non-null  int64
 1   game    25990 non-null  int64
 2   type    25990 non-null  str  
dtypes: int64(2), str(1)
memory usage: 812.2 KB


Есть проблема, в датасете оказалось так, что у одной игры несколько таких площадок и айди дублируются, но с разными площадками, и при попытке смерджить, добавится много ненужных строк, поэтому нужно сначала аггрегировать, датасет уменьшится, и получатся много пропусков в мердженном датасете. Тем не менее, на данном этапе, мы с этим уже ничего не сделаем. Соединим по айди все площадки, и запишем их через запятую, далее можно мердж

In [15]:
df_web = (df_web_final[["game", "type"]].drop_duplicates(["game","type"]).groupby("game", as_index=False)["type"].agg(", ".join))
df_web

,game,type
0,527,"YouTube, Facebook"
1,612,"Community Wiki, Discord, Steam"
2,857,Steam
3,858,"GOG, Wikipedia, Community Wiki"
4,885,Epic
...,...,...
12859,393456,"Facebook, YouTube, Official Website"
12860,393467,Steam
12861,393587,Steam
12862,393954,Steam


In [16]:
df_parsed_2 =df_parsed_1.merge(df_web, left_on="igdb_id", right_on="game",how="left")
df_parsed_2 = df_parsed_2.drop(columns= {"game"})
df_parsed_2

,steam_id,game_name,game_description_snippet,game_price,found_game_price,all_language_reviews_type,all_language_reviews_count,has_russian_reviews,all_russian_reviews_type,all_russian_reviews_count,steam_app_url,support_ru_region,igdb_id,type
0,1313940,My Holiday 2,This is the second part of the long holiday of...,259 руб.,1.0,Очень положительные,106.0,0.0,NaN,NaN,https://store.steampowered.com/app/1313940/,1.0,150866,NaN
1,1902870,Vertical Kingdom,Возродите былое величие Империи в этой карточн...,710 руб.,1.0,В основном положительные,111.0,0.0,NaN,NaN,https://store.steampowered.com/app/1902870/,1.0,202602,"GOG, Nintendo"
2,1071940,Streamer's Life,"Игра в жанре RPG-Simulation, в которой вам пре...",NaN,0.0,В основном положительные,139.0,0.0,NaN,NaN,https://store.steampowered.com/app/1071940/,1.0,124447,NaN
3,1706570,Chupacabras: Night Hunt,Hunt the chupacabras alone or with your friend...,99 руб.,1.0,В основном положительные,124.0,0.0,NaN,NaN,https://store.steampowered.com/app/1706570/,1.0,163904,NaN
4,1085150,Golf Defied,"Физика и механики нового поколения, мультиплее...",Бесплатные,1.0,Смешанные,175.0,0.0,NaN,NaN,https://store.steampowered.com/app/1085150/,1.0,118634,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25694,1547900,Active Mummy,"Active Mummy — динамичный 2D-платформер, в кот...",210 руб.,1.0,Положительные,20.0,0.0,NaN,NaN,https://store.steampowered.com/app/1547900/,1.0,156747,NaN
25695,2238360,You Have No Time,Rev up your inner speedster in this high-octan...,165 руб.,1.0,Положительные,13.0,0.0,NaN,NaN,https://store.steampowered.com/app/2238360/,1.0,244854,NaN
25696,1490030,Cyber Dodge,Cyber Dodge is a skill-based Runner and a Die ...,61 руб.,1.0,Обзоров пользователей: 3,3.0,0.0,NaN,NaN,https://store.steampowered.com/app/1490030/,1.0,159989,NaN
25697,1855190,Vektor Z,Vektor Z brings fast and furious retro space b...,280 руб.,1.0,Обзоров пользователей: 2,2.0,0.0,NaN,NaN,https://store.steampowered.com/app/1855190/,1.0,186672,NaN


In [17]:
df_parsed_2.isna().sum()

steam_id                          0
game_name                         0
game_description_snippet        522
game_price                     5894
found_game_price                954
all_language_reviews_type      1320
all_language_reviews_count     1320
has_russian_reviews             954
all_russian_reviews_type      24165
all_russian_reviews_count     24165
steam_app_url                   954
support_ru_region               954
igdb_id                           0
type                          12820
dtype: int64

Перейдем к следующему датасету

In [18]:
df_games_final.info()
df_games_final

<class 'pandas.DataFrame'>
RangeIndex: 25697 entries, 0 to 25696
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id                  25697 non-null  int64  
 1   first_release_date  25363 non-null  float64
 2   name                25697 non-null  str    
 3   rating              10121 non-null  float64
 4   rating_count        10121 non-null  float64
 5   total_rating        10555 non-null  float64
 6   total_rating_count  10555 non-null  float64
dtypes: float64(5), int64(1), str(1)
memory usage: 1.4 MB


,id,first_release_date,name,rating,rating_count,total_rating,total_rating_count
0,2999,1.454371e+09,Cobalt,80.000000,2.0,67.500000,4.0
1,9177,1.461024e+09,Pollen,50.000000,4.0,55.250000,6.0
2,9216,1.592438e+09,P.A.M.E.L.A.,60.000000,1.0,60.000000,1.0
3,9222,1.399334e+09,Ascendant,60.000000,2.0,52.500000,3.0
4,9233,1.400112e+09,Fearless Fantasy,73.343883,6.0,73.343883,6.0
...,...,...,...,...,...,...,...
25692,321906,1.724371e+09,Floodcore,NaN,NaN,NaN,NaN
25693,326343,1.725494e+09,Skoof Simulator,NaN,NaN,NaN,NaN
25694,343580,1.611792e+09,Gladiator Manager,NaN,NaN,NaN,NaN
25695,345765,NaN,Until None Remain,NaN,NaN,NaN,NaN


Зная, из документации IGDB, что тип данных у first_release_year - это Unix Time Stamp, модифицируем дату 

In [19]:
df_games_final["first_release_date"] = pd.to_datetime(df_games_final["first_release_date"], unit= "s").dt.date
df_games_final #норм, мерджим пока что

,id,first_release_date,name,rating,rating_count,total_rating,total_rating_count
0,2999,2016-02-02,Cobalt,80.000000,2.0,67.500000,4.0
1,9177,2016-04-19,Pollen,50.000000,4.0,55.250000,6.0
2,9216,2020-06-18,P.A.M.E.L.A.,60.000000,1.0,60.000000,1.0
3,9222,2014-05-06,Ascendant,60.000000,2.0,52.500000,3.0
4,9233,2014-05-15,Fearless Fantasy,73.343883,6.0,73.343883,6.0
...,...,...,...,...,...,...,...
25692,321906,2024-08-23,Floodcore,NaN,NaN,NaN,NaN
25693,326343,2024-09-05,Skoof Simulator,NaN,NaN,NaN,NaN
25694,343580,2021-01-28,Gladiator Manager,NaN,NaN,NaN,NaN
25695,345765,NaT,Until None Remain,NaN,NaN,NaN,NaN


In [20]:
df_parsed_3 =df_parsed_2.merge(df_games_final, left_on="igdb_id", right_on="id",how="left")
df_parsed_3 = df_parsed_3.drop(columns= {"id", "name"})
df_parsed_3

,steam_id,game_name,game_description_snippet,game_price,found_game_price,all_language_reviews_type,all_language_reviews_count,has_russian_reviews,all_russian_reviews_type,all_russian_reviews_count,steam_app_url,support_ru_region,igdb_id,type,first_release_date,rating,rating_count,total_rating,total_rating_count
0,1313940,My Holiday 2,This is the second part of the long holiday of...,259 руб.,1.0,Очень положительные,106.0,0.0,NaN,NaN,https://store.steampowered.com/app/1313940/,1.0,150866,NaN,2020-10-05,NaN,NaN,NaN,NaN
1,1902870,Vertical Kingdom,Возродите былое величие Империи в этой карточн...,710 руб.,1.0,В основном положительные,111.0,0.0,NaN,NaN,https://store.steampowered.com/app/1902870/,1.0,202602,"GOG, Nintendo",2024-04-15,NaN,NaN,NaN,NaN
2,1071940,Streamer's Life,"Игра в жанре RPG-Simulation, в которой вам пре...",NaN,0.0,В основном положительные,139.0,0.0,NaN,NaN,https://store.steampowered.com/app/1071940/,1.0,124447,NaN,2019-07-31,NaN,NaN,NaN,NaN
3,1706570,Chupacabras: Night Hunt,Hunt the chupacabras alone or with your friend...,99 руб.,1.0,В основном положительные,124.0,0.0,NaN,NaN,https://store.steampowered.com/app/1706570/,1.0,163904,NaN,2021-08-20,NaN,NaN,NaN,NaN
4,1085150,Golf Defied,"Физика и механики нового поколения, мультиплее...",Бесплатные,1.0,Смешанные,175.0,0.0,NaN,NaN,https://store.steampowered.com/app/1085150/,1.0,118634,NaN,2019-03-11,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25734,1547900,Active Mummy,"Active Mummy — динамичный 2D-платформер, в кот...",210 руб.,1.0,Положительные,20.0,0.0,NaN,NaN,https://store.steampowered.com/app/1547900/,1.0,156747,NaN,2021-02-28,NaN,NaN,NaN,NaN
25735,2238360,You Have No Time,Rev up your inner speedster in this high-octan...,165 руб.,1.0,Положительные,13.0,0.0,NaN,NaN,https://store.steampowered.com/app/2238360/,1.0,244854,NaN,2023-11-09,NaN,NaN,NaN,NaN
25736,1490030,Cyber Dodge,Cyber Dodge is a skill-based Runner and a Die ...,61 руб.,1.0,Обзоров пользователей: 3,3.0,0.0,NaN,NaN,https://store.steampowered.com/app/1490030/,1.0,159989,NaN,2021-02-14,NaN,NaN,NaN,NaN
25737,1855190,Vektor Z,Vektor Z brings fast and furious retro space b...,280 руб.,1.0,Обзоров пользователей: 2,2.0,0.0,NaN,NaN,https://store.steampowered.com/app/1855190/,1.0,186672,NaN,2022-01-14,NaN,NaN,NaN,NaN


Теперь датасет df_popular

In [21]:
df_popular_final['id'].duplicated().sum()

np.int64(11)

In [22]:
df_popular_final[df_popular_final.duplicated(keep=False)] 

,id,game_id,value,external_popularity_source
3632,4088737,19243,1.559722e-06,IGDB
3633,729240,19243,1.568905e-06,Steam
3634,3636534,19243,1.099148e-06,Steam
3635,3636874,19243,1.165209e-06,Steam
3636,2506773,19243,0.000000e+00,Steam
19531,4088737,19243,1.559722e-06,IGDB
19532,729240,19243,1.568905e-06,Steam
19533,3636534,19243,1.099148e-06,Steam
19534,3636874,19243,1.165209e-06,Steam
19535,2506773,19243,0.000000e+00,Steam


In [23]:
df_popular_final[df_popular_final[['game_id']].duplicated(keep=False)] 

,id,game_id,value,external_popularity_source
0,4728328,2999,3.738891e-06,IGDB
1,3496139,2999,1.559722e-05,IGDB
2,297772,2999,3.139584e-06,IGDB
3,3609432,2999,5.110636e-06,Steam
4,2493228,2999,0.000000e+00,Steam
...,...,...,...,...
25995,973954,105548,1.907201e-05,Steam
25996,4764411,105753,9.651292e-08,Twitch
25997,3613242,105753,9.928405e-07,Steam
25998,2495109,105753,0.000000e+00,Steam


In [24]:
df_popular_final[df_popular_final[['game_id', 'external_popularity_source']].duplicated(keep=False)] 

,id,game_id,value,external_popularity_source
0,4728328,2999,3.738891e-06,IGDB
1,3496139,2999,1.559722e-05,IGDB
2,297772,2999,3.139584e-06,IGDB
3,3609432,2999,5.110636e-06,Steam
4,2493228,2999,0.000000e+00,Steam
...,...,...,...,...
25994,3613327,105548,4.840097e-06,Steam
25995,973954,105548,1.907201e-05,Steam
25997,3613242,105753,9.928405e-07,Steam
25998,2495109,105753,0.000000e+00,Steam


In [25]:
df_popular_final['game_id'].nunique() 

4531

Заметим, что хотя и строк довольно много, фактически это все данные для 4531 игры, так что сгруппируем датасет по играм, а также источнику, т.к. для одной игры может быть описана информация с 1 и более источника

In [26]:
df_popular_sum_groupby_source = df_popular_final.groupby(['game_id', 'external_popularity_source'])['value'].sum().reset_index().sort_values(by=['game_id', 'value'], ascending=[True, False])
df_popular_sum_groupby_source

,game_id,external_popularity_source,value
0,111,IGDB,1.007682e-03
1,111,Steam,7.299126e-04
2,111,Twitch,8.010572e-06
3,232,IGDB,6.735236e-04
4,232,Steam,5.761652e-04
...,...,...,...
8595,111924,IGDB,1.559722e-06
8596,111924,Steam,8.353179e-07
8598,112008,Steam,5.410056e-06
8597,112008,IGDB,3.904340e-06


In [27]:
df_popular_sum_groupby_source['external_popularity_source'].value_counts()

external_popularity_source
Steam     4270
IGDB      3893
Twitch     437
Name: count, dtype: int64

У нас всего 3 уникальных источника. Выделим в отдельные столбцы все уникальные значения источников для каждой представленной там игры. И уже эту таблицу будем мерджить с общим датасетом.

In [28]:
df_popular_pivot = df_popular_sum_groupby_source.pivot(index='game_id', columns='external_popularity_source', values='value').reset_index()

df_popular_pivot = df_popular_pivot.rename(columns={'IGDB': 'popularity_igdb', 'Steam': 'popularity_steam', 'Twitch': 'popularity_twitch'})

In [29]:
df_popular_pivot

external_popularity_source,game_id,popularity_igdb,popularity_steam,popularity_twitch
0,111,0.001008,7.299126e-04,8.010572e-06
1,232,0.000674,5.761652e-04,1.592463e-06
2,527,0.000341,6.608975e-04,9.168727e-07
3,532,0.000152,1.075411e-04,3.860517e-07
4,576,0.000250,8.033005e-04,1.703453e-05
...,...,...,...,...
4526,111099,NaN,1.565991e-07,NaN
4527,111872,NaN,6.264701e-07,NaN
4528,111924,0.000002,8.353179e-07,NaN
4529,112008,0.000004,5.410056e-06,NaN


Смерджим с общим датасетом

In [30]:
df_final = df_parsed_3.merge(df_popular_pivot, left_on='igdb_id', right_on='game_id', how='left')
df_final = df_final.drop(columns={'game_id'})

In [31]:
df_final

,steam_id,game_name,game_description_snippet,game_price,found_game_price,all_language_reviews_type,all_language_reviews_count,has_russian_reviews,all_russian_reviews_type,all_russian_reviews_count,...,igdb_id,type,first_release_date,rating,rating_count,total_rating,total_rating_count,popularity_igdb,popularity_steam,popularity_twitch
0,1313940,My Holiday 2,This is the second part of the long holiday of...,259 руб.,1.0,Очень положительные,106.0,0.0,NaN,NaN,...,150866,NaN,2020-10-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1902870,Vertical Kingdom,Возродите былое величие Империи в этой карточн...,710 руб.,1.0,В основном положительные,111.0,0.0,NaN,NaN,...,202602,"GOG, Nintendo",2024-04-15,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1071940,Streamer's Life,"Игра в жанре RPG-Simulation, в которой вам пре...",NaN,0.0,В основном положительные,139.0,0.0,NaN,NaN,...,124447,NaN,2019-07-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1706570,Chupacabras: Night Hunt,Hunt the chupacabras alone or with your friend...,99 руб.,1.0,В основном положительные,124.0,0.0,NaN,NaN,...,163904,NaN,2021-08-20,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1085150,Golf Defied,"Физика и механики нового поколения, мультиплее...",Бесплатные,1.0,Смешанные,175.0,0.0,NaN,NaN,...,118634,NaN,2019-03-11,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25734,1547900,Active Mummy,"Active Mummy — динамичный 2D-платформер, в кот...",210 руб.,1.0,Положительные,20.0,0.0,NaN,NaN,...,156747,NaN,2021-02-28,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25735,2238360,You Have No Time,Rev up your inner speedster in this high-octan...,165 руб.,1.0,Положительные,13.0,0.0,NaN,NaN,...,244854,NaN,2023-11-09,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25736,1490030,Cyber Dodge,Cyber Dodge is a skill-based Runner and a Die ...,61 руб.,1.0,Обзоров пользователей: 3,3.0,0.0,NaN,NaN,...,159989,NaN,2021-02-14,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25737,1855190,Vektor Z,Vektor Z brings fast and furious retro space b...,280 руб.,1.0,Обзоров пользователей: 2,2.0,0.0,NaN,NaN,...,186672,NaN,2022-01-14,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Далее нам предстоит подготовить этот датасет для анализа

In [32]:
df_final.to_csv('../../data/df_final.csv', index=False)